# Week 2 Mini Project: Smart Transportation Fleet Performance & Maintenance Monitoring System
**Course**: MACSE502 - Programming for Data Science Lab  
**Student Name**: VEDANT NIMKAR  
**Registration Number**: 26MML0045  
**Date**: 16-07-2026 | **Faculty**: Dr. Rajasekhara Babu M  

---
## 1. Executive Summary & Problem Formulation
Logistics and commercial transportation companies manage large fleets of commercial trucks and haulage vehicles. Data captured from telematics sensors contains missing coordinates, intermittent network dropouts, and noisy telemetry. This project implements a rigorous data cleaning, missing value imputation, type casting, and normalization pipeline using **Pandas**. Furthermore, the **Mini Project Extension** builds a **Fleet Performance & Maintenance Monitoring System** that flags fuel anomalies, audits route fuel expenditures, and computes a multi-attribute predictive maintenance urgency index.

### Objectives
1. Ingest raw vehicular telematics logs containing missing identifiers, routes, and fuel consumption values.
2. Execute core preprocessing: mean imputation, dropping corrupt rows, data type casting, and Min-Max scaling.
3. Implement the **Mini Project Extension**: inefficiency anomaly detection, route-level fuel variance auditing, composite maintenance risk scoring, and a 4-panel visual fleet dashboard.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 15)
print('Pandas version:', pd.__version__)

## 2. Ingestion of Raw Telematics Data with Missing Values

In [ ]:
csv_path = '../data/w2_transport_fleet_data.csv'
if not os.path.exists(csv_path):
    csv_path = 'data/w2_transport_fleet_data.csv'
df_raw = pd.read_csv(csv_path)
print(f'Raw dataset shape: {df_raw.shape}')
print('\nMissing values per column:')
print(df_raw.isnull().sum())
df_raw.head(8)

## 3. Core Preprocessing & Cleaning Pipeline

In [ ]:
# Step 1 & 2: Mean imputation for FuelConsumption
mean_fuel = df_raw['FuelConsumption'].mean()
df_clean = df_raw.copy()
df_clean['FuelConsumption'] = df_clean['FuelConsumption'].fillna(mean_fuel)
print(f'Imputed missing FuelConsumption with fleet mean: {mean_fuel:.2f} L/100km')

# Step 3: Remove records with missing VehicleID or Route
df_clean = df_clean.dropna(subset=['VehicleID', 'Route']).reset_index(drop=True)
print(f'Retained {len(df_clean)} validated operational vehicles.')

# Step 4: Convert FuelConsumption to integer type
df_clean['FuelConsumption_Int'] = df_clean['FuelConsumption'].round().astype(int)

# Step 5: Min-Max Normalization
f_min = df_clean['FuelConsumption'].min()
f_max = df_clean['FuelConsumption'].max()
df_clean['Fuel_Normalized'] = (df_clean['FuelConsumption'] - f_min) / (f_max - f_min)
display(df_clean[['VehicleID', 'Route', 'FuelConsumption', 'FuelConsumption_Int', 'Fuel_Normalized']].head())

## 4. Mini Project Extension: Fleet Performance Monitoring & Predictive Maintenance

In [ ]:
# 1. Identify Fuel-Inefficient Vehicles (Threshold >= 0.70)
inefficient = df_clean[df_clean['Fuel_Normalized'] >= 0.70].copy()
inefficient['ExcessBurnPct'] = ((inefficient['FuelConsumption'] - mean_fuel) / mean_fuel) * 100
print(f'Identified {len(inefficient)} fuel-inefficient haulage units:')
display(inefficient[['VehicleID', 'Route', 'FuelConsumption', 'ExcessBurnPct', 'Fuel_Normalized']])

# 2. Route Financial & Energy Audit
fuel_rate = 96.50  # INR per Litre
df_clean['TripLiters'] = (df_clean['DistanceKM'] / 100) * df_clean['FuelConsumption']
df_clean['FuelCostINR'] = df_clean['TripLiters'] * fuel_rate
route_audit = df_clean.groupby('Route').agg(
    ActiveVehicles=('VehicleID', 'count'),
    AvgFuel=('FuelConsumption', 'mean'),
    StdFuel=('FuelConsumption', 'std'),
    TotalDistanceKM=('DistanceKM', 'sum'),
    TotalFuelCostINR=('FuelCostINR', 'sum')
).reset_index()
route_audit['CostPerKM'] = route_audit['TotalFuelCostINR'] / route_audit['TotalDistanceKM']
display(route_audit.sort_values(by='AvgFuel', ascending=False))

# 3. Predictive Maintenance Risk Score
norm_eh = (df_clean['EngineHours'] - df_clean['EngineHours'].min()) / (df_clean['EngineHours'].max() - df_clean['EngineHours'].min())
norm_d = (df_clean['DistanceKM'] - df_clean['DistanceKM'].min()) / (df_clean['DistanceKM'].max() - df_clean['DistanceKM'].min())
df_clean['MaintenanceRiskScore'] = 0.50 * df_clean['Fuel_Normalized'] + 0.30 * norm_eh + 0.20 * norm_d

def triage_status(score):
    if score >= 0.70: return 'Immediate Overhaul Required'
    elif score >= 0.45: return 'Scheduled Maintenance Due'
    else: return 'Optimal Operational Health'
df_clean['MaintenanceStatus'] = df_clean['MaintenanceRiskScore'].apply(triage_status)
print('\nFleet Maintenance Status Triage:')
print(df_clean['MaintenanceStatus'].value_counts())

## 5. Visual Fleet Performance Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Smart Transportation Fleet Performance & Maintenance Monitoring System\nVEDANT NIMKAR (26MML0045) | Week 2 Mini Project', fontsize=15, weight='bold', y=0.98)

# Panel 1: Fuel Consumption Distribution
sns.histplot(df_clean['FuelConsumption'], kde=True, ax=axes[0, 0], color='#1f77b4', bins=8, edgecolor='black')
axes[0, 0].axvline(mean_fuel, color='green', linestyle='--', linewidth=2, label=f'Fleet Mean ({mean_fuel:.1f} L/100km)')
thresh_val = f_min + 0.70 * (f_max - f_min)
axes[0, 0].axvline(thresh_val, color='red', linestyle='-', linewidth=2, label=f'Inefficiency Threshold ({thresh_val:.1f})')
axes[0, 0].set_title('Fleet Fuel Consumption Distribution & Anomaly Boundary', weight='bold')
axes[0, 0].set_xlabel('Fuel Consumption (L/100km)')
axes[0, 0].legend()

# Panel 2: Route-Wise Fuel with Error Bars
axes[0, 1].bar(route_audit['Route'], route_audit['AvgFuel'], yerr=route_audit['StdFuel'].fillna(0),
               capsize=5, color=sns.color_palette('muted', len(route_audit)), edgecolor='black', alpha=0.85)
axes[0, 1].set_title('Route-Wise Mean Fuel Burn with Operational Variance', weight='bold')
axes[0, 1].set_ylabel('Avg Fuel Consumption (L/100km)')
axes[0, 1].tick_params(axis='x', rotation=15)

# Panel 3: Scatter Plot of Maintenance Risk
status_palette = {'Immediate Overhaul Required': '#d62728', 'Scheduled Maintenance Due': '#ff7f0e', 'Optimal Operational Health': '#2ca02c'}
sns.scatterplot(data=df_clean, x='FuelConsumption', y='MaintenanceRiskScore', hue='MaintenanceStatus', palette=status_palette, s=120, edgecolor='black', ax=axes[1, 0])
axes[1, 0].axhline(0.70, color='red', linestyle=':', label='Overhaul Threshold (0.70)')
axes[1, 0].set_title('Predictive Maintenance Risk vs. Fuel Consumption', weight='bold')
axes[1, 0].set_xlabel('Fuel Consumption (L/100km)')
axes[1, 0].set_ylabel('Composite Maintenance Risk Index')
axes[1, 0].legend(loc='upper left', frameon=True)

# Panel 4: Maintenance Status Breakdown
m_counts = df_clean['MaintenanceStatus'].value_counts()
axes[1, 1].pie(m_counts, labels=m_counts.index, autopct='%1.1f%%', colors=[status_palette[k] for k in m_counts.index], startangle=90, wedgeprops=dict(width=0.45, edgecolor='black'))
axes[1, 1].set_title('Fleet Health & Maintenance Triage Distribution', weight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()